# AHC015 T4 x2 parallel training smoke test

GPU T4 x2とInternetを有効にし、`GITHUB_TOKEN`と`WANDB_API_KEY`へのSecret accessを許可して実行する。並列rollout・PPO・並列評価を、PPO minibatchに端数が出ない1,024 episodesで1 iterationだけ検証する。元の学習checkpointは変更しない。

In [ ]:
import torch

assert torch.cuda.is_available()
assert torch.cuda.device_count() == 2, "Select GPU T4 x2"
for index in range(2):
    print(index, torch.cuda.get_device_name(index), torch.cuda.get_device_capability(index))
    assert torch.cuda.get_device_capability(index) == (7, 5)

In [ ]:
import base64
import os
import subprocess
from pathlib import Path

from kaggle_secrets import UserSecretsClient

repo_dir = Path("/kaggle/working/ahc-ml")
expected_commit = "96bcb834fdadb8f9bbd3f47e250206f9546c2c32"
assert not repo_dir.exists()
github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
credentials = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env = os.environ.copy()
git_env["GIT_CONFIG_COUNT"] = "1"
git_env["GIT_CONFIG_KEY_0"] = "http.extraHeader"
git_env["GIT_CONFIG_VALUE_0"] = f"Authorization: Basic {credentials}"
try:
    subprocess.run(
        [
            "git", "clone", "--branch", "feature/ahc015-teacher",
            "--single-branch", "https://github.com/e1jirou/ahc-ml.git", str(repo_dir),
        ],
        check=True,
        env=git_env,
    )
finally:
    del github_token, credentials, git_env
subprocess.run(
    ["git", "-C", str(repo_dir), "checkout", "--detach", expected_commit],
    check=True,
)
actual_commit = subprocess.check_output(
    ["git", "-C", str(repo_dir), "rev-parse", "HEAD"], text=True
).strip()
assert actual_commit == expected_commit
print("Repository commit:", actual_commit)

In [ ]:
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "torchview==0.2.7"],
    check=True,
)

In [ ]:
import os
from pathlib import Path

import wandb
from kaggle_secrets import UserSecretsClient

os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
checkpoint_dir = Path("/kaggle/working/checkpoints/ppo-20260826-000916")
artifact = wandb.Api().artifact(
    "eijirou-personal/ahc-ml/ppo-20260826-000916-training-checkpoint:latest",
    type="model",
)
downloaded_dir = Path(artifact.download(root=checkpoint_dir))
checkpoint_path = downloaded_dir / "best-training.pt"
assert checkpoint_path.is_file()
print("Checkpoint:", checkpoint_path)

In [ ]:
import os
import subprocess
import sys

benchmark_env = os.environ.copy()
benchmark_env["PYTHONPATH"] = str(repo_dir / "python")
benchmark_env["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

smoke_output = Path("/kaggle/working/parallel-training-smoke-output")
print("\n===== parallel training smoke test =====", flush=True)
subprocess.run(
    [
        sys.executable,
        "-m",
        "examples.ahc015.python.train",
        "--config",
        "examples/ahc015/config.toml",
        "--seed",
        "15029",
        "--device",
        "cuda",
        "--resume",
        str(checkpoint_path),
        "--iterations",
        "4",
        "--rollout-episodes",
        "1024",
        "--batch-size",
        "1024",
        "--micro-batch-size",
        "512",
        "--data-parallel",
        "--rollout-processes",
        "2",
        "--epochs",
        "1",
        "--evaluation-episodes",
        "64",
        "--max-hours",
        "0.25",
        "--wandb-mode",
        "disabled",
        "--output-dir",
        str(smoke_output),
        "--experiment-log",
        "/kaggle/working/parallel-training-smoke-experiments.md",
    ],
    cwd=repo_dir,
    env=benchmark_env,
    check=True,
)
print("Parallel training smoke test completed")